# Concept Explanations Visualization

This notebook visualizes pre-computed concept-based explanations.

It loads cached artifacts (interpretations, global importances, local importances)
from the `data/` directory and renders them using interpreto's `plot_concepts`.

## Parameters

In [ ]:
# ---------------------------------------------------------------------------
# Configuration: adjust these to select the dataset / method / interpretation
# ---------------------------------------------------------------------------

# Dataset short name: "RT", "GE", "BIOS", "AG", "IMDB", "E", "HE"
DATASET_ABBREV = "RT"

# Concept method: "seminmf", "ica", "kmeans", "pca", "svd", "batchtopk_sae", "vanilla_sae", "neurons"
METHOD = "seminmf"

# Interpretation: "topk" or "llm"
INTERPRETATION = "topk"

# Number of concepts ratio (nb_concepts = nb_classes * ratio)
NB_CONCEPTS_RATIO = 3

# Classes subset index (within DATASET_CLASSES_SUBSETS for the dataset)
CLASSES_SUBSET_IDX = 0

# Seed for local explanations (sample selection)
SEED = 0

# Number of samples per seed (used to locate the local_elements cache file)
NB_SAMPLES = 5

## Imports and path resolution

In [ ]:
import sys
from pathlib import Path

# Add repo root to path so we can import utils
REPO_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(REPO_ROOT))

import json
import torch
from interpreto import plot_concepts

from utils.data import (
    MODELS_DATASETS,
    ABBREVIATIONS,
    DATASET_CLASSES_NAMES,
    DATASET_CLASSES_SUBSETS,
    get_save_root,
)
from utils.concepts import CONCEPT_METHOD_NAMES

In [ ]:
# Resolve full dataset/model names from abbreviation
dataset_name = next(k for k, v in ABBREVIATIONS["datasets"].items() if v == DATASET_ABBREV)
model_name = next(k for k, v in MODELS_DATASETS.items() if v == dataset_name)

# Classes subset
classes_subset = DATASET_CLASSES_SUBSETS[dataset_name][CLASSES_SUBSET_IDX]
classes_names = [DATASET_CLASSES_NAMES[dataset_name][i] for i in classes_subset]
nb_classes = len(classes_subset)
nb_concepts = nb_classes * NB_CONCEPTS_RATIO

# Paths
save_root = REPO_ROOT / get_save_root(model_name)
method_dir_name = CONCEPT_METHOD_NAMES[METHOD]
if METHOD == "neurons":
    # NeuronsAs uses hidden_dim as nb_concepts (768 for BERT/RoBERTa)
    concept_dirs = list(save_root.glob(f"concept_models/{method_dir_name}_nc*"))
    assert len(concept_dirs) == 1, f"Expected 1 NeuronsAs dir, found {len(concept_dirs)}: {concept_dirs}"
    concept_dir = concept_dirs[0]
else:
    concept_dir = save_root / f"concept_models/{method_dir_name}_nc{nb_concepts}"

print(f"Dataset: {dataset_name}")
print(f"Model: {model_name}")
print(f"Classes: {classes_names} (indices {classes_subset})")
print(f"Concept dir: {concept_dir}")
print(f"Exists: {concept_dir.exists()}")

## Load concept interpretations

In [ ]:
# Load interpretations (topk words or LLM labels)
if INTERPRETATION == "topk":
    # Try the standard filename first, then the BIOS variant
    interp_path = concept_dir / "topk_interpretations.json"
    if not interp_path.exists():
        interp_path = concept_dir / "topk_words_interpretations.json"
elif INTERPRETATION == "llm":
    interp_path = concept_dir / "llm_interpretations.json"
else:
    raise ValueError(f"Unknown interpretation: {INTERPRETATION}")

assert interp_path.exists(), f"Interpretation file not found: {interp_path}"

with open(interp_path) as f:
    raw_interpretations = json.load(f)

# Convert keys to int. Values are either a list of words (topk) or a single string (llm).
concepts_interpretation: dict[int, str] = {}
for k, v in raw_interpretations.items():
    if isinstance(v, list):
        concepts_interpretation[int(k)] = ", ".join(v)
    else:
        concepts_interpretation[int(k)] = v

print(f"Loaded {len(concepts_interpretation)} concept interpretations from {interp_path.name}")
for idx in list(concepts_interpretation.keys())[:5]:
    print(f"  C{idx}: {concepts_interpretation[idx]}")

## Load global importances

In [ ]:
importances_path = concept_dir / "importances.pt"
assert importances_path.exists(), f"Importances not found: {importances_path}"

raw_importances = torch.load(importances_path, map_location="cpu")

# importances.pt stores a list of tensors (one per sample used for gradient computation).
# Stack and average to get shape (nb_classes, nb_concepts)
global_importances = torch.stack(raw_importances).abs().squeeze().mean(0)

# If the dataset uses a class subset, select only those class rows
if global_importances.ndim == 2 and global_importances.shape[0] > nb_classes:
    global_importances = global_importances[classes_subset]

print(f"Global importances shape: {global_importances.shape}")
print(f"  Expected: ({nb_classes}, {nb_concepts if METHOD != 'neurons' else '?'})")

## Visualize global concept importances

Interactive plot: click on classes to see their most important concepts.

In [ ]:
# Build labels dict: concept_id -> list of interpretation words
concepts_labels = {}
for k, v in concepts_interpretation.items():
    concepts_labels[k] = v.split(", ") if ", " in v else [v]

plot_concepts(
    classes_names=classes_names,
    concepts_importances=global_importances,
    concepts_labels=concepts_labels,
)

## Load local importances for selected samples

In [ ]:
# Load the all_local_importances tensor (one per test sample)
local_imp_path = concept_dir / "all_local_importances.pt"
assert local_imp_path.exists(), f"Local importances not found: {local_imp_path}"

all_local_importances = torch.load(local_imp_path, map_location="cpu")
print(f"Loaded local importances for {len(all_local_importances)} test samples")
print(f"  Each tensor shape: {all_local_importances[0].shape}")

In [ ]:
# Load sample selection for this seed
classes_str = "-".join(str(c) for c in classes_subset)
local_elements_path = save_root / f"local_elements_classes_{classes_str}_n{NB_SAMPLES}.json"
assert local_elements_path.exists(), f"Local elements not found: {local_elements_path}"

with open(local_elements_path) as f:
    local_elements = json.load(f)

seed_data = local_elements[str(SEED)]
sample_indices = seed_data["indices"]
sample_texts = seed_data["texts"]
sample_predictions = seed_data["predictions"]

print(f"Seed {SEED}: {len(sample_indices)} samples")
for i, (idx, text, pred) in enumerate(zip(sample_indices, sample_texts, sample_predictions)):
    pred_name = classes_names[classes_subset.index(pred)] if pred in classes_subset else str(pred)
    print(f"  [{i}] idx={idx}, pred={pred_name}: {text[:80]}...")

## Visualize local concept importances per sample

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select a sample to visualize
SAMPLE_IDX = 0  # index within the seed's sample list

sample_global_idx = sample_indices[SAMPLE_IDX]
local_imp = all_local_importances[sample_global_idx].squeeze()  # (nb_classes, nb_concepts)

# Select only the relevant classes
if local_imp.ndim == 2 and local_imp.shape[0] > nb_classes:
    local_imp = local_imp[classes_subset]

print(f"Sample {SAMPLE_IDX} (global idx {sample_global_idx}):")
print(f"  Text: {sample_texts[SAMPLE_IDX][:120]}")
print(f"  Local importances shape: {local_imp.shape}")

In [ ]:
# Heatmap of local concept importances for this sample
fig, ax = plt.subplots(figsize=(max(8, nb_concepts * 0.5), nb_classes * 0.8 + 2))

imp_np = local_imp.numpy()
im = ax.imshow(imp_np, aspect="auto", cmap="RdBu_r")

ax.set_yticks(range(nb_classes))
ax.set_yticklabels(classes_names)

# X-axis: concept labels (truncated)
concept_ids = sorted(concepts_interpretation.keys())[:local_imp.shape[1]]
x_labels = [f"C{cid}" for cid in range(local_imp.shape[1])]
ax.set_xticks(range(local_imp.shape[1]))
ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)

ax.set_title(f"Local concept importances — Sample {SAMPLE_IDX}\n\"{sample_texts[SAMPLE_IDX][:60]}...\"")
plt.colorbar(im, ax=ax, label="Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Top concepts for the predicted class of this sample
pred_class_idx = classes_subset.index(sample_predictions[SAMPLE_IDX]) if sample_predictions[SAMPLE_IDX] in classes_subset else 0
pred_importances = local_imp[pred_class_idx]

top_k = min(10, len(pred_importances))
top_indices = torch.argsort(pred_importances.abs(), descending=True)[:top_k]

print(f"\nTop {top_k} concepts for predicted class '{classes_names[pred_class_idx]}':")
print(f"{'Concept':<8} {'Importance':>12}  {'Interpretation'}")
print("-" * 60)
for idx in top_indices:
    cid = idx.item()
    imp_val = pred_importances[cid].item()
    interp = concepts_interpretation.get(cid, "N/A")
    print(f"C{cid:<6} {imp_val:>+12.4f}  {interp[:50]}")

## Compare global importances across all samples in seed

In [ ]:
# Stack local importances for all samples in this seed
seed_local_imps = []
for global_idx in sample_indices:
    imp = all_local_importances[global_idx].squeeze()
    if imp.ndim == 2 and imp.shape[0] > nb_classes:
        imp = imp[classes_subset]
    seed_local_imps.append(imp)

seed_local_imps = torch.stack(seed_local_imps)  # (nb_samples, nb_classes, nb_concepts)
print(f"Stacked local importances shape: {seed_local_imps.shape}")

# Average local importances across samples (gives a view similar to global)
avg_local = seed_local_imps.abs().mean(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].imshow(global_importances.numpy(), aspect="auto", cmap="Reds")
axes[0].set_title("Global importances (all test samples)")
axes[0].set_yticks(range(nb_classes))
axes[0].set_yticklabels(classes_names)
axes[0].set_xlabel("Concept index")

axes[1].imshow(avg_local.numpy(), aspect="auto", cmap="Reds")
axes[1].set_title(f"Avg local importances (seed {SEED}, {len(sample_indices)} samples)")
axes[1].set_yticks(range(nb_classes))
axes[1].set_yticklabels(classes_names)
axes[1].set_xlabel("Concept index")

plt.tight_layout()
plt.show()